In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [69]:
import pandas as pd

df = pd.read_csv('/kaggle/input/datasets/harshrishi7/master-dataseet/master_dataset.csv')

In [70]:
df['datetime'] = pd.to_datetime(df['datetime'])

In [71]:
df['hour'] = df['datetime'].dt.hour
df['weekday'] = df['datetime'].dt.day_name()
df['is_weekend'] = df['weekday'].isin(['Saturday','Sunday'])
df['is_weekend'] =df['is_weekend'].astype(int)

In [72]:
df['month'] = df['datetime'].dt.month

In [73]:
def peak_period(hour):
    if 7 <= hour <= 10:
        return 'Morning Peak'
    elif 17 <= hour <= 21:
        return 'Evening Peak'
    elif 11 <= hour <= 16:
        return 'Shoulder'
    else:
        return 'Off-Peak'
df['peak_period'] = df['hour'].apply(peak_period)

In [74]:
df['revenue'] = df['volume'] * df['price']

In [75]:
df['utilization_rate'] = (
    df['occupancy'] / df['count']
)


In [76]:
df['congestion_score'] = (
    0.7 * df['occupancy']
    +
    0.3 * df['duration']
)

In [77]:
df['high_congestion'] = (
   df['utilization_rate'] > 0.8
).astype(int)
df['low_utilization'] = (
  df['utilization_rate'] < 0.3
).astype(int)

In [78]:
df['fast_charger_ratio'] = (
    df['fast_count'] / df['count']
)

In [79]:
def charger_type(row):
    if row['fast_count'] > row['slow_count']:
        return 'Fast Dominant'
    else:
        return 'Slow Dominant'

df['charger_category'] = df.apply(
    charger_type,
    axis=1
)

In [80]:
df['CBD_area'] = df['CBD'].map({
    1:'CBD',
    0:'Non-CBD'
})

In [82]:
df['available_time'] = df['count'] * 5
df['charger_utilization_rate'] = (
   df['duration'] / df['available_time']
)

In [84]:
df['revenue_per_session'] = (
   df['revenue'] /
    (df['occupancy'] + 1)
)

In [85]:
df['energy_cost_per_kwh'] = (
    df['revenue'] /
    (df['volume'] + 1)
)

In [87]:
df['queue_length_proxy'] = (
   df['occupancy']
    *
   df['duration']
) /df['count']

In [89]:
df['occupancy_density'] = (
    df['occupancy'] /
    (df['area'] + 1)
)

In [91]:
df.head()

,timestamp,station_id,occupancy,duration,volume,price,datetime,num,count,fast_count,...,low_utilization,fast_charger_ratio,charger_category,CBD_area,available_time,charger_utilization_rate,revenue_per_session,energy_cost_per_kwh,queue_length_proxy,occupancy_density
0,1,102,12,0.49,2.858333,0.924,2022-06-19 00:00:00,1,30,3,...,0,0.1,Slow Dominant,Non-CBD,150,0.003267,0.203162,0.684518,0.196,7.017544
1,2,102,12,0.75,4.375000,0.924,2022-06-19 00:05:00,1,30,3,...,0,0.1,Slow Dominant,Non-CBD,150,0.005000,0.310962,0.752093,0.300,7.017544
2,3,102,12,0.75,4.375000,0.924,2022-06-19 00:10:00,1,30,3,...,0,0.1,Slow Dominant,Non-CBD,150,0.005000,0.310962,0.752093,0.300,7.017544
3,4,102,12,0.75,4.375000,0.924,2022-06-19 00:15:00,1,30,3,...,0,0.1,Slow Dominant,Non-CBD,150,0.005000,0.310962,0.752093,0.300,7.017544
4,5,102,12,0.75,4.375000,0.924,2022-06-19 00:20:00,1,30,3,...,0,0.1,Slow Dominant,Non-CBD,150,0.005000,0.310962,0.752093,0.300,7.017544


In [92]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2134080 entries, 0 to 2134079
Data columns (total 35 columns):
 #   Column                    Dtype         
---  ------                    -----         
 0   timestamp                 int64         
 1   station_id                int64         
 2   occupancy                 int64         
 3   duration                  float64       
 4   volume                    float64       
 5   price                     float64       
 6   datetime                  datetime64[ns]
 7   num                       int64         
 8   count                     int64         
 9   fast_count                int64         
 10  slow_count                int64         
 11  area                      float64       
 12  lon                       float64       
 13  la                        float64       
 14  CBD                       int64         
 15  dynamic_pricing           int64         
 16  hour                      int32         
 17  weekday 

In [94]:
df.to_csv(
    'featured_master_dataset.csv',
    index=False
)